In [1]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import root_mean_squared_error

from utils.data_splitter import DataSplitter
from utils.wrangle_model_data import wrangle_ml

df = pd.read_csv('data/commodity_prices.csv')
df = wrangle_ml(df)

date_dict = {
    'train_start': "2023-06-01", 'train_end': "2025-06-30",
    'valid_start': "2025-07-01", 'valid_end': "2025-07-31",
    'test_start': "2025-08-01", 'test_end': "2025-08-18"
}

thresholds = {
    'train': 250,
    'valid': 10,
    'test': 5
}

splitter = DataSplitter(df, date_dict, thresholds)
train_df, valid_df, test_df = splitter.run()

features = [col for col in train_df.columns if col not in ['log_Modal_Price', 'log_Modal_Price_filled', 'Arrival_Date']]
target_col = "log_Modal_Price_filled"

X = train_df[features].copy()
y = train_df[target_col].copy()

categorical_cols = ['Product_Type', 'Commodity', 'Variety_Type', 'Market', 'Season', 'Market_Season', 'Variety_Type', 'Product_Month']

for c in categorical_cols:
    X[c] = X[c].astype('category')

In [ ]:
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

def objective(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "n_estimators": 10000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    rmses = []
    for train_idx, val_idx in tscv.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols)
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_cols)

        model = lgb.train(
            params,
            dtrain,
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=0) 
            ]
        )
        preds = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, preds)
        rmses.append(rmse)
 
    return np.mean(rmses)

study = optuna.create_study(direction="minimize", study_name="lgbm_ts", sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=100, show_progress_bar=False) 

print("Best trial:", study.best_trial.params)

[I 2025-09-11 15:04:07,854] A new study created in memory with name: lgbm_ts


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[297]	valid_0's rmse: 0.179212
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[390]	valid_0's rmse: 0.104312
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[278]	valid_0's rmse: 0.118745
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[297]	valid_0's rmse: 0.105022
Training until validation scores don't improve for 100 rounds


[I 2025-09-11 15:04:30,537] Trial 0 finished with value: 0.12210596771794571 and parameters: {'learning_rate': 0.02468755662051332, 'num_leaves': 157, 'max_depth': 8, 'min_child_samples': 73, 'subsample': 0.53498093521982, 'colsample_bytree': 0.8106090055190661, 'reg_alpha': 2.247567005264056, 'reg_lambda': 0.001313006540546148}. Best is trial 0 with value: 0.12210596771794571.


Early stopping, best iteration is:
[318]	valid_0's rmse: 0.103238
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[64]	valid_0's rmse: 0.183845
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[61]	valid_0's rmse: 0.104602
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[54]	valid_0's rmse: 0.120963
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[60]	valid_0's rmse: 0.108181
Training until validation scores don't improve for 100 rounds


[I 2025-09-11 15:04:48,121] Trial 1 finished with value: 0.1249417442194074 and parameters: {'learning_rate': 0.08752424234996359, 'num_leaves': 216, 'max_depth': 15, 'min_child_samples': 174, 'subsample': 0.6664428260166262, 'colsample_bytree': 0.6091931456956776, 'reg_alpha': 7.884126068385365e-05, 'reg_lambda': 3.5284792103029357e-08}. Best is trial 0 with value: 0.12210596771794571.


...

Early stopping, best iteration is:
[116]	valid_0's rmse: 0.102304
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[232]	valid_0's rmse: 0.163437
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[640]	valid_0's rmse: 0.105205
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[109]	valid_0's rmse: 0.119942
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[133]	valid_0's rmse: 0.105847
Training until validation scores don't improve for 100 rounds


[I 2025-09-11 15:16:40,443] Trial 98 finished with value: 0.1191363157931516 and parameters: {'learning_rate': 0.03987912308184379, 'num_leaves': 46, 'max_depth': 4, 'min_child_samples': 24, 'subsample': 0.6164734152758307, 'colsample_bytree': 0.6455224467893795, 'reg_alpha': 0.027620777510198598, 'reg_lambda': 8.240025875673036e-06}. Best is trial 85 with value: 0.1188170244856972.


Early stopping, best iteration is:
[140]	valid_0's rmse: 0.101251
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[159]	valid_0's rmse: 0.176684
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[479]	valid_0's rmse: 0.10344
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[149]	valid_0's rmse: 0.119735
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[131]	valid_0's rmse: 0.104893
Training until validation scores don't improve for 100 rounds


[I 2025-09-11 15:16:47,622] Trial 99 finished with value: 0.12120862342950796 and parameters: {'learning_rate': 0.038993469285251776, 'num_leaves': 57, 'max_depth': 5, 'min_child_samples': 23, 'subsample': 0.5703138069566308, 'colsample_bytree': 0.6430977786788923, 'reg_alpha': 0.011545781161119205, 'reg_lambda': 4.159038997010491e-05}. Best is trial 85 with value: 0.1188170244856972.


Early stopping, best iteration is:
[152]	valid_0's rmse: 0.101292
Best trial: {'learning_rate': 0.04394712160197687, 'num_leaves': 89, 'max_depth': 4, 'min_child_samples': 64, 'subsample': 0.6932149359549723, 'colsample_bytree': 0.6071046036207595, 'reg_alpha': 0.05489072905466115, 'reg_lambda': 6.133073516239546e-06}


In [4]:
print(study.best_params)
print(study.best_value)

{'learning_rate': 0.04394712160197687, 'num_leaves': 89, 'max_depth': 4, 'min_child_samples': 64, 'subsample': 0.6932149359549723, 'colsample_bytree': 0.6071046036207595, 'reg_alpha': 0.05489072905466115, 'reg_lambda': 6.133073516239546e-06}
0.1188170244856972


In [ ]:
best_params = study.best_trial.params
best_params.update({
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "n_estimators": 10000
})

final_model = lgb.LGBMRegressor(**best_params)
final_model.fit(X, y)